In [121]:
import json
import matplotlib.pyplot as plt

In [122]:
real_folder_name = "./data-json/iscx"
# fake_folder_name = "./data-json/iscx_netdiffusion"
fake_folder_name = "./result/iscx"

type_name = "email"
real_name = real_folder_name + "/" + type_name + ".json"
fake_name = fake_folder_name + "/" + type_name + ".json"

In [123]:
def extract_dirs(name):
    with open(name, 'r') as f:
        json_data = json.load(f)
    dirs = []
    for flow in json_data:
        flow_pkt = []
        sery = flow['series']
        for pkt in sery:
            flow_pkt.append(1 if pkt['pkt_len'] > 0 else 0)
        dirs.append(flow_pkt)
        
    return dirs

In [124]:
def extract_flags(name):
    with open(name, 'r') as f:
        json_data = json.load(f)
    flags = []
    for flow in json_data:
        flow_pkt = []
        sery = flow['series']
        for pkt in sery:
            flow_pkt.append(pkt['flags'])
        flags.append(flow_pkt)
        
    return flags

In [125]:
from collections import Counter


# ============================================================
# TCP flag definitions
# ============================================================

FIN = 0x01
SYN = 0x02
RST = 0x04
PSH = 0x08
ACK = 0x10
URG = 0x20


# ============================================================
# Flag classification
# ============================================================

def classify_flag(flag):
    """
    Convert raw TCP flag bitmask into a coarse event type.

    Note:
        flag == 0 is handled at flow level as UDP,
        so it should normally not reach the TCP checker.
    """

    if flag == 0:
        return "EMPTY"

    if flag & RST:
        return "RST"

    if (flag & SYN) and (flag & ACK):
        return "SYN_ACK"

    if (flag & SYN) and not (flag & ACK):
        return "SYN"

    if flag & FIN:
        return "FIN"

    if (flag & PSH) and (flag & ACK):
        return "PSH_ACK"

    if flag & ACK:
        return "ACK"

    return "OTHER"

In [126]:
# ============================================================
# Flow-level protocol classification
# ============================================================

def classify_flow_protocol(flags):
    """
    According to the current data encoding:

        all flags == 0
            -> UDP

        mixture of 0 and non-zero
            -> MIXED / invalid encoding

        all flags != 0
            -> TCP
    """

    if not flags:
        return "EMPTY_FLOW"

    has_zero = any(f == 0 for f in flags)
    has_nonzero = any(f != 0 for f in flags)

    if has_zero and not has_nonzero:
        return "UDP"

    if has_zero and has_nonzero:
        return "MIXED"

    return "TCP"


In [127]:
# ============================================================
# Direction-aware TCP flow checker
# ============================================================

def check_tcp_flow(flags, dirs):
    """
    Direction-aware, conservative TCP flag/state consistency checker.

    Returns:
        valid_events
        invalid_events
        unknown_events
    """

    assert len(flags) == len(dirs)

    state = "UNKNOWN"

    # Direction that initiated the TCP connection.
    initiator_dir = None

    valid_events = []
    invalid_events = []
    unknown_events = []

    previous_event = None
    previous_dir = None

    for i, (flag, direction) in enumerate(zip(flags, dirs)):

        event = classify_flag(flag)

        # ----------------------------------------------------
        # OTHER
        # ----------------------------------------------------

        if event == "OTHER":
            unknown_events.append({
                "index": i,
                "flag": flag,
                "dir": direction,
                "event": event,
                "state": state,
                "reason": "Unrecognized TCP flag combination"
            })
            previous_event = event
            previous_dir = direction
            continue

        # ----------------------------------------------------
        # SYN
        # ----------------------------------------------------

        if event == "SYN":

            if state == "UNKNOWN":
                state = "SYN_SENT"
                initiator_dir = direction

                valid_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "Initial SYN"
                })

            elif state == "SYN_SENT" and direction == initiator_dir:
                # SYN retransmission
                valid_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "SYN retransmission"
                })

            elif state == "ESTABLISHED":
                # Could be an unusual observation, retransmission,
                # or a new connection whose boundaries are not known.
                unknown_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "SYN observed after connection establishment"
                })

            else:
                unknown_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "SYN cannot be uniquely interpreted in current state"
                })

            previous_event = event
            previous_dir = direction
            continue

        # ----------------------------------------------------
        # SYN + ACK
        # ----------------------------------------------------

        if event == "SYN_ACK":

            if (
                state == "SYN_SENT"
                and initiator_dir is not None
                and direction != initiator_dir
            ):
                state = "SYN_RECEIVED"

                valid_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "Reverse-direction SYN-ACK"
                })

            elif (
                state == "SYN_RECEIVED"
                and initiator_dir is not None
                and direction != initiator_dir
            ):
                # SYN-ACK retransmission
                valid_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "SYN-ACK retransmission"
                })

            elif state == "UNKNOWN":
                # Capture may start at SYN-ACK.
                initiator_dir = 1 - direction
                state = "SYN_RECEIVED"

                unknown_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "Capture begins at SYN-ACK; initiator inferred"
                })

            else:
                unknown_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "SYN-ACK cannot be uniquely interpreted"
                })

            previous_event = event
            previous_dir = direction
            continue

        # ----------------------------------------------------
        # ACK
        # ----------------------------------------------------

        if event == "ACK":

            if (
                state == "SYN_RECEIVED"
                and initiator_dir is not None
                and direction == initiator_dir
            ):
                state = "ESTABLISHED"

                valid_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "Final ACK of handshake"
                })

            elif state == "ESTABLISHED":
                valid_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "ACK in established connection"
                })

            elif state == "UNKNOWN":
                # Mid-stream capture.
                state = "ESTABLISHED"

                unknown_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "ACK without observed handshake; likely mid-stream capture"
                })

            elif state == "SYN_SENT":
                # Without sequence numbers we cannot know whether
                # this is a valid ACK associated with the handshake.
                unknown_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "ACK observed before SYN-ACK"
                })

            else:
                unknown_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "ACK cannot be uniquely interpreted"
                })

            previous_event = event
            previous_dir = direction
            continue

        # ----------------------------------------------------
        # PSH + ACK
        # ----------------------------------------------------

        if event == "PSH_ACK":

            if state == "ESTABLISHED":

                valid_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "PSH-ACK in established connection"
                })

            elif (
                state == "SYN_RECEIVED"
                and initiator_dir is not None
                and direction == initiator_dir
            ):
                # Conservative interpretation:
                # first data packet may also serve as the final ACK.
                state = "ESTABLISHED"

                valid_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "PSH-ACK completes handshake and carries data"
                })

            elif state == "UNKNOWN":
                # Mid-stream capture.
                state = "ESTABLISHED"

                unknown_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "PSH-ACK without observed handshake; likely mid-stream capture"
                })

            elif state == "SYN_SENT":
                unknown_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "PSH-ACK observed before SYN-ACK"
                })

            else:
                unknown_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "PSH-ACK cannot be uniquely interpreted"
                })

            previous_event = event
            previous_dir = direction
            continue

        # ----------------------------------------------------
        # FIN
        # ----------------------------------------------------

        if event == "FIN":

            if state == "ESTABLISHED":

                state = "FIN_WAIT"

                valid_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "FIN from established connection"
                })

            elif state == "FIN_WAIT":

                valid_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "Additional FIN during connection close"
                })

            elif state == "UNKNOWN":

                unknown_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "FIN without observed connection establishment"
                })

            else:

                unknown_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "FIN cannot be uniquely interpreted"
                })

            previous_event = event
            previous_dir = direction
            continue

        # ----------------------------------------------------
        # RST
        # ----------------------------------------------------

        if event == "RST":

            if state in {
                "SYN_SENT",
                "SYN_RECEIVED",
                "ESTABLISHED",
                "FIN_WAIT"
            }:

                state = "CLOSED"

                valid_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "RST closes connection"
                })

            elif state == "UNKNOWN":

                # A capture can begin with RST.
                state = "CLOSED"

                unknown_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "RST observed without prior connection state"
                })

            elif state == "CLOSED":

                valid_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "Repeated RST after closure"
                })

            else:

                unknown_events.append({
                    "index": i,
                    "event": event,
                    "dir": direction,
                    "state": state,
                    "reason": "RST cannot be uniquely interpreted"
                })

            previous_event = event
            previous_dir = direction
            continue

        # ----------------------------------------------------
        # Fallback
        # ----------------------------------------------------

        unknown_events.append({
            "index": i,
            "event": event,
            "dir": direction,
            "state": state,
            "reason": "Unhandled event"
        })

        previous_event = event
        previous_dir = direction

    return valid_events, invalid_events, unknown_events

In [128]:
# ============================================================
# Evaluate all flows
# ============================================================

def evaluate_all_flows(flags_all, dirs_all, show_examples=10):

    assert len(flags_all) == len(dirs_all)

    total_flows = len(flags_all)

    udp_flows = 0
    tcp_flows = 0
    mixed_flows = 0
    empty_flows = 0

    tcp_valid = 0
    tcp_invalid = 0
    tcp_unknown = 0

    tcp_evaluable = 0
    tcp_consistent = 0

    mixed_details = []
    invalid_details = []

    unknown_reason_counter = Counter()

    for flow_id, (flags, dirs) in enumerate(
        zip(flags_all, dirs_all)
    ):

        protocol = classify_flow_protocol(flags)

        # ----------------------------------------------------
        # Empty flow
        # ----------------------------------------------------

        if protocol == "EMPTY_FLOW":

            empty_flows += 1
            continue

        # ----------------------------------------------------
        # UDP
        # ----------------------------------------------------

        if protocol == "UDP":

            udp_flows += 1
            continue

        # ----------------------------------------------------
        # Mixed 0/non-zero flow
        # ----------------------------------------------------

        if protocol == "MIXED":

            mixed_flows += 1

            mixed_details.append({
                "flow_id": flow_id,
                "flags": flags,
                "dirs": dirs,
            })

            continue

        # ----------------------------------------------------
        # TCP
        # ----------------------------------------------------

        tcp_flows += 1

        valid, invalid, unknown = check_tcp_flow(
            flags,
            dirs
        )

        tcp_valid += len(valid)
        tcp_invalid += len(invalid)
        tcp_unknown += len(unknown)

        # Count reason of unknown events.
        for item in unknown:
            unknown_reason_counter[item["reason"]] += 1

        if len(invalid) == 0:
            tcp_consistent += 1

        # A TCP flow is considered evaluable when it contains
        # at least one recognizable TCP event.
        recognizable = any(
            classify_flag(f) != "OTHER"
            for f in flags
        )

        if recognizable:
            tcp_evaluable += 1

        if len(invalid) > 0:
            invalid_details.append({
                "flow_id": flow_id,
                "flags": flags,
                "dirs": dirs,
                "valid": valid,
                "invalid": invalid,
                "unknown": unknown,
            })

    # ========================================================
    # Metrics
    # ========================================================

    # UDP validity:
    # under the current encoding, every all-zero flow is valid.
    udp_validity = (
        udp_flows / (udp_flows + mixed_flows)
        if (udp_flows + mixed_flows) > 0
        else 1.0
    )

    # Encoding consistency:
    # only mixed 0/non-zero flows are considered encoding errors.
    encoding_consistency = (
        1.0 - mixed_flows / total_flows
        if total_flows > 0
        else 1.0
    )

    # TCP transition validity excludes unknown transitions.
    tcp_transition_validity = (
        tcp_valid / (tcp_valid + tcp_invalid)
        if (tcp_valid + tcp_invalid) > 0
        else 1.0
    )

    tcp_flow_consistency = (
        tcp_consistent / tcp_flows
        if tcp_flows > 0
        else 1.0
    )

    tcp_flow_coverage = (
        tcp_evaluable / tcp_flows
        if tcp_flows > 0
        else 0.0
    )

    # ========================================================
    # Print summary
    # ========================================================

    print("=" * 70)
    print("TCP / UDP Flag Consistency Evaluation")
    print("=" * 70)

    print(f"Total flows:              {total_flows}")
    print(f"UDP flows:                {udp_flows}")
    print(f"TCP flows:                {tcp_flows}")
    print(f"Mixed 0/non-zero flows:   {mixed_flows}")
    print(f"Empty flows:              {empty_flows}")

    print()
    print("-" * 70)
    print("UDP / Encoding")
    print("-" * 70)

    print(f"UDP flow validity:        {udp_validity:.4f}")
    print(f"Encoding consistency:     {encoding_consistency:.4f}")

    print()
    print("-" * 70)
    print("TCP")
    print("-" * 70)

    print(f"TCP flow coverage:        {tcp_flow_coverage:.4f}")
    print(f"Valid transitions:        {tcp_valid}")
    print(f"Invalid transitions:      {tcp_invalid}")
    print(f"Unknown transitions:      {tcp_unknown}")
    print(f"Transition validity:      {tcp_transition_validity:.4f}")
    print(f"Flow consistency:         {tcp_flow_consistency:.4f}")

    print()
    print("-" * 70)
    print("Unknown transition reasons")
    print("-" * 70)

    if unknown_reason_counter:
        for reason, count in unknown_reason_counter.most_common():
            print(f"{count:8d}  {reason}")
    else:
        print("None")

    # ========================================================
    # Show mixed examples
    # ========================================================

    if mixed_details:
        print()
        print("-" * 70)
        print(f"Mixed-flow examples (first {show_examples})")
        print("-" * 70)

        for item in mixed_details[:show_examples]:
            print(
                f"Flow {item['flow_id']}: "
                f"flags={item['flags']}"
            )

    # ========================================================
    # Show invalid examples
    # ========================================================

    if invalid_details:
        print()
        print("-" * 70)
        print(f"Invalid TCP examples (first {show_examples})")
        print("-" * 70)

        for item in invalid_details[:show_examples]:

            print(
                f"\nFlow {item['flow_id']}"
            )

            print(
                "  flags:",
                item["flags"]
            )

            print(
                "  dirs: ",
                item["dirs"]
            )

            for inv in item["invalid"][:5]:
                print(
                    "  INVALID:",
                    inv
                )

    print("=" * 70)

    return {
        "total_flows": total_flows,
        "udp_flows": udp_flows,
        "tcp_flows": tcp_flows,
        "mixed_flows": mixed_flows,
        "empty_flows": empty_flows,

        "udp_validity": udp_validity,
        "encoding_consistency": encoding_consistency,

        "tcp_flow_coverage": tcp_flow_coverage,
        "tcp_valid": tcp_valid,
        "tcp_invalid": tcp_invalid,
        "tcp_unknown": tcp_unknown,
        "tcp_transition_validity": tcp_transition_validity,
        "tcp_flow_consistency": tcp_flow_consistency,

        "unknown_reasons": unknown_reason_counter,

        "mixed_details": mixed_details,
        "invalid_details": invalid_details,
    }

In [ ]:
real_flags = extract_flags(real_name)
real_dirs = extract_dirs(real_name)

generated_flags = extract_flags(fake_name)
generated_dirs = extract_dirs(fake_name)

print("\n\n")
print("#" * 80)
print("# REAL DATA")
print("#" * 80)

real_result = evaluate_all_flows(
    real_flags,
    real_dirs
)

print("\n\n")
print("#" * 80)
print("# GENERATED DATA")
print("#" * 80)

generated_result = evaluate_all_flows(
    generated_flags,
    generated_dirs
)




################################################################################
# GENERATED DATA
################################################################################
TCP / UDP Flag Consistency Evaluation
Total flows:              298
UDP flows:                155
TCP flows:                143
Mixed 0/non-zero flows:   0
Empty flows:              0

----------------------------------------------------------------------
UDP / Encoding
----------------------------------------------------------------------
UDP flow validity:        1.0000
Encoding consistency:     1.0000

----------------------------------------------------------------------
TCP
----------------------------------------------------------------------
TCP flow coverage:        1.0000
Valid transitions:        1498
Invalid transitions:      0
Unknown transitions:      774
Transition validity:      1.0000
Flow consistency:         1.0000

----------------------------------------------------------------------
Unk